In [0]:
df_select_columns = spark.read \
                .option("header", True) \
                .option("inferSchema", True) \
                .csv("/Volumes/rentsight/airbnb-infos/01-raw/listings.csv")

df_select_columns = df_select_columns.select("id", "neighbourhood", "latitude", "longitude", "room_type", "price", "minimum_nights", "number_of_reviews", "last_review", "reviews_per_month", "availability_365")

display(df_select_columns)

In [0]:
#Ajuste no Id
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window = Window.orderBy(F.lit(1))  
df_new_id = df_select_columns.withColumn("id", F.row_number().over(window))

display(df_new_id)


In [0]:
# Vendo valores por coluna

for col_name in df_new_id.columns:
    print(f"\n=== Coluna: {col_name} ===")
    df_new_id.groupBy(col_name).count().orderBy("count", ascending=False).show(truncate=False)


In [0]:
# Corrigindo Schema

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType

df_safe = df_new_id \
    .withColumn("price", F.expr("try_cast(price AS DOUBLE)")) \
    .withColumn("minimum_nights", F.expr("try_cast(minimum_nights AS INT)")) \
    .withColumn("number_of_reviews", F.expr("try_cast(number_of_reviews AS INT)")) \
    .withColumn("last_review", F.expr("try_cast(last_review AS DATE)")) \
    .withColumn("reviews_per_month", F.expr("try_cast(reviews_per_month AS DOUBLE)")) \
    .withColumn("availability_365", F.expr("try_cast(availability_365 AS INT)")) \
    .withColumn("room_type", F.expr("try_cast(room_type AS STRING)")) \
    .withColumn("latitude", F.expr("try_cast(latitude AS DOUBLE)")) \
    .withColumn("longitude", F.expr("try_cast(longitude AS DOUBLE)"))
display(df_safe)

In [0]:
from pyspark.sql.functions import col

display(df_safe.orderBy(col("price").desc()).limit(40))

In [0]:
display( df_safe
        .filter(col("price").isNotNull())
        .orderBy(col("price").asc())
        .limit(40))

In [0]:
display(df_safe.select("neighbourhood").distinct())

In [0]:
# ================================
# Room Type: limpar + simular (random)
# ================================

from pyspark.sql.functions import col, lower, trim, when, rand

# Normalização
df_safe = df_safe.withColumn(
    "room_type_norm",
    lower(trim(col("room_type")))
)

# Limpeza semântica (domínio válido)
df_safe = df_safe.withColumn(
    "room_type_clean",
    when(col("room_type_norm") == "entire home/apt", "Entire home/apt")
    .when(col("room_type_norm") == "private room", "Private room")
    .when(col("room_type_norm") == "shared room", "Shared room")
    .when(col("room_type_norm") == "hotel room", "Hotel room")
    .otherwise(None)
)

# Sorteio controlado (1 rand por linha)
df_safe = df_safe.withColumn("r", rand())

df_safe = df_safe.withColumn(
    "room_type_simulated",
    when(col("room_type_clean").isNotNull(), col("room_type_clean"))
    .when(col("r") < 0.25, "Entire home/apt")
    .when(col("r") < 0.50, "Private room")
    .when(col("r") < 0.75, "Shared room")
    .otherwise("Hotel room")
)

# Limpeza final (fica só o simulated)
df_safe_rooms = (
    df_safe
        .drop("room_type")
        .drop("room_type_norm")
        .drop("room_type_clean")
        .drop("r")
)

# Preview
display(
   df_safe_rooms
)


In [0]:
df_finish_layer = df_safe_rooms

from pyspark.sql.functions import col, count, when

total = df_finish_layer.count()

completeness = df_finish_layer.select([
    (count(when(col(c).isNull(), c)) / total).alias(c)
    for c in df_finish_layer.columns
])

completeness.show()


In [0]:
from pyspark.sql.functions import col, count, when, lit, round

df = df_finish_layer
total = df.count()

rows = []
for c in df.columns:
    nulls = df.select(count(when(col(c).isNull(), 1)).alias("nulls")).collect()[0]["nulls"]
    rows.append((c, nulls, total - nulls, nulls / total))

dq = spark.createDataFrame(rows, ["column", "null_count", "non_null_count", "null_ratio"]) \
          .withColumn("null_pct", round(col("null_ratio") * 100, 2)) \
          .orderBy(col("null_ratio").desc())

display(dq)


In [0]:
df_finish_layer.write.mode("overwrite").option("header", True).parquet("/Volumes/rentsight/airbnb-infos/02-silver/listings_silver_rj")